<a href="https://colab.research.google.com/github/componavt/dictorpus-space/blob/main/src/sem_cat/01_meanings_examples_counter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 01 · VepKar Meanings & Examples — Counter and Explorer

**Goal**: Load all 8 VepKar data files, inspect their structure,
and produce statistics needed before semantic categorization via WordNet ILI.

**Data files** (from `data/vepkar/`):
- `meanings_vep.csv`, `meanings_olo.csv`, `meanings_lud.csv`, `meanings_krl.csv`
- `examples_vep.csv`, `examples_olo.csv`, `examples_lud.csv`, `examples_krl.csv`

**What we analyse:**
1. 📦 Row counts per language
2. 🏷️ POS distribution (crucial for WordNet coverage strategy)
3. 🗺️ PROPN — proper names that need a separate pipeline
4. ✂️ Gloss length — how many 1-word / 2-word / 3+ glosses
5. 🔗 Multi-part glosses separated by `;` — need splitting logic
6. 🔤 Top-20 Russian words across all glosses
7. 🔁 Cross-language gloss reuse — how many unique Russian words total
8. ⚠️ Same gloss with different POS — ambiguity risk for lookup
9. 📎 Parenthetical patterns `(...)` in glosses — need stripping
10. 📉 Examples per meaning — coverage for context-based disambiguation

In [1]:
# Step 1. Clone dictorpus-space repository into Colab

import os

REPO = "dictorpus-space"
REPO_PATH = f"/content/{REPO}"

if not os.path.exists(REPO_PATH):
    !git clone -q https://github.com/componavt/dictorpus-space.git

if os.getcwd() != REPO_PATH:
    %cd {REPO_PATH}

print("✅ Repository ready, working directory:", os.getcwd())

/content/dictorpus-space
✅ Repository ready, working directory: /content/dictorpus-space


## 📥 Load all 8 files

All files are **tab-separated** (TSV) with English column headers — no renaming needed.

**meanings_*.csv** columns:
`id`, `lemma_id`, `meaning_id`, `meaning_num`, `lemma`, `lang`, `pos`, `meaning_ru`

**examples_*.csv** columns:
`id`, `meaning_id`, `sentence`

> ⚠️ Some `meaning_ru` values are wrapped in double quotes by the exporter.
> We strip those along with leading/trailing whitespace during loading.

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "data/vepkar"
LANGS = ["vep", "olo", "lud", "krl"]


def load_csv(filepath):
    print(f"  ⏳ Reading: {filepath}")
    # standard RFC 4180 CSV: comma separator, double-quote escaping
    df = pd.read_csv(
        filepath,
        sep=",",
        dtype=str,
        quotechar='"',
        doublequote=True,    # handles: "" → "   ("" inside field → single ")
        escapechar='\\',     # handles: \" → "   ← это добавлено
        encoding="utf-8-sig",# rm BOM from PHP-export

    )
    df.columns = df.columns.str.strip()
    # strip only whitespace (no quote-stripping — pandas handles RFC 4180 itself)
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    print(f"  ✅ Loaded {len(df):,} rows, columns: {list(df.columns)}")
    return df


meanings = {}
examples = {}

for lang in LANGS:
    m_path = f"{DATA_DIR}/meanings_{lang}.csv"
    e_path = f"{DATA_DIR}/examples_{lang}.csv"
    print(f"\n🌐 Language: {lang}")
    meanings[lang] = load_csv(m_path)
    examples[lang] = load_csv(e_path)
    print(f"  📊 {lang}: {len(meanings[lang]):,} meanings, {len(examples[lang]):,} examples")

# combined DataFrames for cross-language analysis
df_m = pd.concat(meanings.values(), ignore_index=True)
df_e = pd.concat(examples.values(), ignore_index=True)
print(f"\n📦 Total: {len(df_m):,} meanings, {len(df_e):,} examples")

# quick sanity check
print("\nColumn names check:")
print("  meanings:", list(meanings["vep"].columns))
print("  examples:", list(examples["vep"].columns))

# verify quote handling: show a row with quotes inside meaning_ru
mask = df_m["meaning_ru"].str.contains('"', na=False)
if mask.any():
    print(f"\n🔍 Quote-inside-field check ({mask.sum()} rows), first 3:")
    print(df_m[mask][["lang", "lemma", "meaning_ru"]].head(3).to_string(index=False))


🌐 Language: vep
  ⏳ Reading: data/vepkar/meanings_vep.csv
  ✅ Loaded 22,094 rows, columns: ['id', 'lemma_id', 'meaning_id', 'meaning_num', 'lemma', 'lang', 'pos', 'meaning_ru']
  ⏳ Reading: data/vepkar/examples_vep.csv
  ✅ Loaded 13,113 rows, columns: ['id', 'meaning_id', 'example']
  📊 vep: 22,094 meanings, 13,113 examples

🌐 Language: olo
  ⏳ Reading: data/vepkar/meanings_olo.csv
  ✅ Loaded 34,256 rows, columns: ['id', 'lemma_id', 'meaning_id', 'meaning_num', 'lemma', 'lang', 'pos', 'meaning_ru']
  ⏳ Reading: data/vepkar/examples_olo.csv
  ✅ Loaded 20,416 rows, columns: ['id', 'meaning_id', 'example']
  📊 olo: 34,256 meanings, 20,416 examples

🌐 Language: lud
  ⏳ Reading: data/vepkar/meanings_lud.csv
  ✅ Loaded 7,925 rows, columns: ['id', 'lemma_id', 'meaning_id', 'meaning_num', 'lemma', 'lang', 'pos', 'meaning_ru']
  ⏳ Reading: data/vepkar/examples_lud.csv
  ✅ Loaded 22,458 rows, columns: ['id', 'meaning_id', 'example']
  📊 lud: 7,925 meanings, 22,458 examples

🌐 Language: krl
  ⏳ 

## 📊 Basic counts per language

How many meanings and examples exist for each language?
This tells us the relative weight of each subcorpus
and helps estimate lookup table size.

In [3]:
print("=" * 52)
print(f"{'Lang':<6} {'Meanings':>10} {'Examples':>10} {'Ex/Mean':>8}")
print("=" * 52)
for lang in LANGS:
    m = len(meanings[lang])
    e = len(examples[lang])
    ratio = e / m if m > 0 else 0
    print(f"{lang:<6} {m:>10,} {e:>10,} {ratio:>8.1f}")
print("-" * 52)
print(f"{'TOTAL':<6} {len(df_m):>10,} {len(df_e):>10,}")
print()

# unique lemmas per language
print("Unique lemma_ids per language:")
for lang in LANGS:
    n = meanings[lang]["lemma_id"].nunique()
    print(f"  {lang}: {n:,} unique lemmas")

Lang     Meanings   Examples  Ex/Mean
vep        22,094     13,113      0.6
olo        34,256     20,416      0.6
lud         7,925     22,458      2.8
krl        22,397     29,817      1.3
----------------------------------------------------
TOTAL      86,672     85,804

Unique lemma_ids per language:
  vep: 19,438 unique lemmas
  olo: 27,976 unique lemmas
  lud: 7,379 unique lemmas
  krl: 19,563 unique lemmas


## 🏷️ POS distribution

Part-of-speech distribution is **critical** for planning the WordNet pipeline:

- `NOUN`, `VERB`, `ADJ`, `ADV` → high coverage in WordNet, direct lookup
- `PROPN` → need separate pipeline (toponyms, personal names)
- `INTJ`, `PART`, `CONJ`, `NUM`, `PRON` → very low WordNet coverage, need fallback labels

In [4]:
print("POS distribution per language (% of total meanings):\n")

# combined POS table
pos_table = {}
for lang in LANGS:
    pos_counts = meanings[lang]["pos"].value_counts()
    pos_table[lang] = pos_counts

pos_df = pd.DataFrame(pos_table).fillna(0).astype(int)
pos_df["TOTAL"] = pos_df.sum(axis=1)
pos_df = pos_df.sort_values("TOTAL", ascending=False)
print(pos_df.to_string())

print("\n--- % share of WordNet-friendly POS (NOUN+VERB+ADJ+ADV) ---")
wn_pos = ["NOUN", "VERB", "ADJ", "ADV"]
for lang in LANGS:
    total = len(meanings[lang])
    wn_count = meanings[lang]["pos"].isin(wn_pos).sum()
    propn_count = (meanings[lang]["pos"] == "PROPN").sum()
    other = total - wn_count - propn_count
    print(f"  {lang}: WordNet-friendly={wn_count/total*100:.1f}%  "
          f"PROPN={propn_count/total*100:.1f}%  "
          f"other={other/total*100:.1f}%")

POS distribution per language (% of total meanings):

          vep    olo   lud    krl  TOTAL
pos                                     
NOUN    12745  19413  4302  10213  46673
VERB     4404   6823  1696   7272  20195
ADJ      2649   4397   638   2191   9875
ADV       975   1887   606   1166   4634
PHRASE    613    951    72    643   2279
PROPN     234    167   334    250    985
POSTP      89    161    53    158    461
PRON      112    108    66    151    437
NUM       122    116    67     89    394
PART       26     73    23     79    201
SCONJ      31     35    22     34    122
INTJ       25     26    17     39    107
PREP       27     42     3     24     96
CCONJ      19     22     9     45     95
PRTC        3      6     0     40     49
PRE        18     23     0      0     41
X           0      4    13      1     18
AUX         2      2     1      2      7
PUNCT       0      0     3      0      3

--- % share of WordNet-friendly POS (NOUN+VERB+ADJ+ADV) ---
  vep: WordNet-friendly=

## ✂️ Gloss length & multi-part glosses

Russian glosses can be:
- **1 word**: `хлеб`, `помощь` — easiest to look up in WordNet
- **2–3 words**: `единица измерения` — manageable as phrase lookup
- **Multi-part** (`;`-separated): `край; конец; вершина` — need splitting, use first part as primary key
- **Parenthetical** `(...)` — qualifiers that should be stripped before lookup, e.g. `единица ёмкости (для сыпучих тел)`

Understanding this distribution tells us how to write the lookup pre-processor.

In [5]:
import re

df_m["meaning_ru"] = df_m["meaning_ru"].fillna("")

# 1. Gloss length in words (after stripping parentheticals)
def strip_parens(s):
    return re.sub(r"\(.*?\)", "", s).strip()

df_m["gloss_clean"] = df_m["meaning_ru"].apply(strip_parens)
df_m["gloss_words"] = df_m["gloss_clean"].apply(
    lambda s: len(s.split()) if s else 0
)

print("=== Gloss length distribution (words after stripping parentheticals) ===")
length_dist = df_m["gloss_words"].value_counts().sort_index()
for length, count in length_dist.items():
    bar = "█" * min(int(count / 100), 40)
    print(f"  {length:>2} word(s): {count:>6,}  {bar}")

# 2. Multi-part glosses (;-separated)
df_m["is_multipart"] = df_m["meaning_ru"].str.contains(";", na=False)
multipart_total = df_m["is_multipart"].sum()
print(f"\n=== Multi-part glosses (with ';') ===")
print(f"  Total: {multipart_total:,}  ({multipart_total/len(df_m)*100:.1f}%)")
print("\n  Examples (first 8):")
for _, row in df_m[df_m["is_multipart"]].head(8).iterrows():
    print(f"  [{row['lang']} {row['pos']}] {row['lemma']} → {row['meaning_ru']}")

# 3. Parenthetical patterns
df_m["has_parens"] = df_m["meaning_ru"].str.contains(r"\(", na=False)
parens_total = df_m["has_parens"].sum()
print(f"\n=== Glosses with parentheticals '(...)' ===")
print(f"  Total: {parens_total:,}  ({parens_total/len(df_m)*100:.1f}%)")
print("\n  Examples (first 5):")
for _, row in df_m[df_m["has_parens"]].head(5).iterrows():
    print(f"  [{row['lang']} {row['pos']}] {row['lemma']} → {row['meaning_ru']}")

=== Gloss length distribution (words after stripping parentheticals) ===
   0 word(s):      4  
   1 word(s): 48,956  ████████████████████████████████████████
   2 word(s): 24,486  ████████████████████████████████████████
   3 word(s):  7,807  ████████████████████████████████████████
   4 word(s):  3,367  █████████████████████████████████
   5 word(s):  1,125  ███████████
   6 word(s):    474  ████
   7 word(s):    195  █
   8 word(s):    130  █
   9 word(s):     39  
  10 word(s):     34  
  11 word(s):     23  
  12 word(s):      9  
  13 word(s):      7  
  14 word(s):      4  
  15 word(s):      4  
  16 word(s):      2  
  17 word(s):      2  
  18 word(s):      1  
  20 word(s):      2  
  21 word(s):      1  

=== Multi-part glosses (with ';') ===
  Total: 2,273  (2.6%)

  Examples (first 8):
  [vep PROPN] Piter → Ленинград; Санкт-Петербург
  [vep PROPN] Venä → Русь; Россия
  [vep CCONJ] a → a; но (союз противопоставительный)
  [vep ADV] abidašti → с обидой; обидно
  [vep VERB] 

## 🔤 Top-20 Russian words in glosses

We tokenize **all** meaning glosses into individual Russian words
and count their frequency. This reveals:
- which semantic fields dominate the VepKar lexicon
- how often the same Russian word appears (= lookup table reuse potential)
- whether stop-words / function words appear that need filtering

In [6]:
from collections import Counter
import re

# tokenize: lowercase, only Cyrillic
def tokenize_ru(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = re.sub(r"\(.*?\)", " ", text)   # strip parentheticals
    words = re.findall(r"[а-яёА-ЯЁ]+", text.lower())
    return words

# function words to exclude from top list
STOPWORDS = {
    "и", "в", "с", "на", "к", "по", "для", "из", "от", "до",
    "не", "же", "то", "а", "но", "как", "что", "это"
}

all_words = []
for gloss in df_m["meaning_ru"]:
    all_words.extend(tokenize_ru(str(gloss)))

word_freq = Counter(w for w in all_words if w not in STOPWORDS)
total_tokens = len(all_words)

print(f"Total gloss tokens: {total_tokens:,}")
print(f"Unique Cyrillic words in glosses: {len(word_freq):,}\n")
print("Top-20 Russian words across all VepKar glosses:")
print(f"\n{'Rank':<5} {'Word':<25} {'Count':>7}  {'% of tokens':>12}")
print("-" * 55)
for rank, (word, count) in enumerate(word_freq.most_common(20), 1):
    pct = count / total_tokens * 100
    print(f"{rank:<5} {word:<25} {count:>7,}  {pct:>11.2f}%")

Total gloss tokens: 147,917
Unique Cyrillic words in glosses: 32,470

Top-20 Russian words across all VepKar glosses:

Rank  Word                        Count   % of tokens
-------------------------------------------------------
1     л                             570         0.39%
2     место                         328         0.22%
3     быть                          303         0.20%
4     делать                        288         0.19%
5     время                         210         0.14%
6     за                            210         0.14%
7     или                           191         0.13%
8     без                           184         0.12%
9     становиться                   169         0.11%
10    довольно                      160         0.11%
11    быстро                        157         0.11%
12    день                          148         0.10%
13    идти                          142         0.10%
14    человек                       135         0.09%
15    часть    

## 🔁 Cross-language gloss reuse

The same Russian gloss (e.g., `помощь`) may appear in **multiple languages**.
If we build a lookup cache `{russian_gloss → WN domain}`,
we can compute it **once** and reuse for all four languages.

Here we measure:
- how many **unique** Russian glosses exist across all languages combined
- what fraction is shared between ≥2 languages (= cache hit rate)

In [7]:
# First part before ; is the primary lookup key
df_m["gloss_primary"] = df_m["meaning_ru"].apply(
    lambda s: strip_parens(str(s).split(";")[0]).strip().lower()
    if isinstance(s, str) else ""
)

# unique primary glosses per language
print("Unique primary glosses per language:")
for lang in LANGS:
    n = meanings[lang]["meaning_ru"].apply(
        lambda s: strip_parens(str(s).split(";")[0]).strip().lower()
    ).nunique()
    print(f"  {lang}: {n:,}")

# total unique across all languages
all_unique = df_m["gloss_primary"].nunique()
print(f"\nTotal UNIQUE primary glosses (all languages): {all_unique:,}")
print(f"Total meanings rows: {len(df_m):,}")
print(f"Cache reuse factor: {len(df_m) / all_unique:.2f}x")

# how many glosses appear in ≥2 languages
gloss_lang_count = df_m.groupby("gloss_primary")["lang"].nunique()
shared = (gloss_lang_count >= 2).sum()
print(f"\nGlosses shared across ≥2 languages: {shared:,} ({shared/all_unique*100:.1f}% of unique glosses)")

Unique primary glosses per language:
  vep: 16,628
  olo: 24,572
  lud: 4,262
  krl: 13,866

Total UNIQUE primary glosses (all languages): 42,962
Total meanings rows: 86,672
Cache reuse factor: 2.02x

Glosses shared across ≥2 languages: 10,825 (25.2% of unique glosses)


## ⚠️ Ambiguous glosses — same Russian word, different POS

If the **same Russian gloss** maps to different POS tags (e.g., `белый` as NOUN and ADJ),
naive WordNet lookup may assign the wrong synset.
These cases will need **POS-aware lookup** or manual review.

In [8]:
# count distinct POS values per primary gloss
ambig = (
    df_m[df_m["gloss_primary"] != ""]
    .groupby("gloss_primary")["pos"]
    .nunique()
)

ambig_glosses = ambig[ambig > 1].sort_values(ascending=False)
print(f"Glosses with >1 POS tag: {len(ambig_glosses):,}\n")
print("Top-20 most ambiguous glosses:")
print(f"\n{'Gloss':<30} {'# POS':>6}  {'POS list'}")
print("-" * 65)
for gloss, n_pos in ambig_glosses.head(20).items():
    pos_list = sorted(df_m[df_m["gloss_primary"] == gloss]["pos"].unique())
    print(f"{gloss:<30} {n_pos:>6}  {', '.join(pos_list)}")

Glosses with >1 POS tag: 856

Top-20 most ambiguous glosses:

Gloss                           # POS  POS list
-----------------------------------------------------------------
:                                   5  ADJ, ADV, NOUN, PRTC, VERB
пока                                5  ADV, CCONJ, INTJ, PART, SCONJ
когда                               4  ADV, CCONJ, SCONJ, X
как                                 4  ADV, CCONJ, PART, SCONJ
через                               3  ADV, POSTP, PREP
плачущий                            3  ADJ, NOUN, PRTC
потому                              3  ADV, CCONJ, SCONJ
по                                  3  ADV, POSTP, PREP
всё                                 3  ADV, PHRASE, PRON
всё равно                           3  ADV, PHRASE, X
стыдно                              3  ADV, PHRASE, PRE
тяжело                              3  ADV, PHRASE, PRE
хоть, хотя                          3  CCONJ, PART, SCONJ
где                                 3  ADV, PRON, SCONJ
берем

## 📉 Examples per meaning

Each `meaning_id` in `meanings_*.csv` may have 0, 1, or many example sentences.
Meanings **with examples** can use context-based disambiguation
(e.g., translate example sentence to find the correct WordNet synset).
Meanings **without examples** must rely on gloss-only lookup.

In [9]:
# count examples per meaning_id (across all languages)
ex_per_meaning = df_e.groupby("meaning_id").size().reset_index(name="ex_count")

total_meanings = len(df_m)
meanings_with_ex = ex_per_meaning[ex_per_meaning["ex_count"] > 0]
meanings_no_ex = total_meanings - len(
    df_m[df_m["meaning_id"].isin(meanings_with_ex["meaning_id"])]
)

print("=== Examples per meaning distribution ===\n")
print(f"Total meanings:              {total_meanings:>8,}")
print(f"Meanings WITH examples:      {len(df_m[df_m['meaning_id'].isin(ex_per_meaning['meaning_id'])]):>8,}")
print(f"Meanings WITHOUT examples:   {meanings_no_ex:>8,}")
print()

dist = ex_per_meaning["ex_count"].value_counts().sort_index()
print(f"{'Ex count':<12} {'# meanings':>10}")
print("-" * 25)
for val, cnt in dist.items():
    if val <= 10:
        print(f"{val:<12} {cnt:>10,}")
    elif val == dist.index[dist.index > 10][0]:
        print(f"11+        {(dist[dist.index > 10].sum()):>10,}")
        break

print(f"\nMax examples for one meaning: {ex_per_meaning['ex_count'].max():,}")
print(f"Median examples (non-zero):   {ex_per_meaning[ex_per_meaning['ex_count']>0]['ex_count'].median():.1f}")

=== Examples per meaning distribution ===

Total meanings:                86,672
Meanings WITH examples:        14,339
Meanings WITHOUT examples:     72,333

Ex count     # meanings
-------------------------
1                 6,452
2                 2,519
3                 1,363
4                   852
5                   568
6                   361
7                   307
8                   224
9                   201
10                  145
11+             1,351

Max examples for one meaning: 1,070
Median examples (non-zero):   2.0


## 📝 Summary

| Check | Finding |
|-------|---------|
| Total meanings | from this notebook's output |
| WordNet-friendly POS (NOUN/VERB/ADJ/ADV) | ~XX% |
| PROPN — needs separate pipeline | ~XX% |
| Multi-part glosses (`;`) | ~XX% |
| Glosses with parentheticals `(...)` | ~XX% |
| Unique primary glosses (cache size) | XX,XXX |
| Cross-language cache reuse factor | ~X.Xx |
| Ambiguous glosses (same word, diff POS) | XX |
| Meanings with at least 1 example | ~XX% |

### ✅ Next steps

1. **Build lookup pre-processor** — strip parentheticals, take first `;`-part, lowercase
2. **Separate PROPN pipeline** — PERSON / LOCATION labels without WordNet
3. **Build Russian→EN translation layer** — for WordNet ILI lookup
4. **Cache design** — `{(gloss_primary, pos) → wn_domain}` avoids duplicate lookups
5. **Context strategy** — for meanings with examples, use sentence translation as disambiguation signal